In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Train 3-Tier 4-Model Hierarchical LightGBM Sub-Models (`models/train_hierarchical_lightgbm_layers.ipynb`)

This notebook trains all 4 sub-models of the **3-Tier Hierarchical LightGBM Triage Architecture** using the **38-feature uniform input matrix** and **SMOTE upsampling**:

### Sub-Model Layers Trained
- **Layer 1**: Binary LightGBM (ESI 1 Detector)
- **Layer 2**: Binary LightGBM (ESI 2/3 vs ESI 4/5 Specialist)
- **Layer 3A**: Binary LightGBM (ESI 2 vs ESI 3 Specialist)
- **Layer 3B**: Binary LightGBM (ESI 4 vs ESI 5 Specialist)

All trained sub-model RDS files are exported to `deploy/` for pipeline assembly and C transpilation.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Standard Libraries & Define Helper Functions
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pROC)
  library(lightgbm)
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) config_path <- "config/triage_conf.json"
config <- fromJSON(config_path)
set.seed(config$training$random_state)
# Native R Base Helpers
stratified_partition <- function(y, p, seed = 42) {
  set.seed(seed)
  idx_list <- split(seq_along(y), y)
  train_idx <- unlist(lapply(idx_list, function(indices) {
    sample(indices, size = max(1, round(length(indices) * p)))
  }))
  return(sort(train_idx))
}
fit_scaler <- function(df_train, cols) {
  means <- colMeans(df_train[, cols, drop = FALSE], na.rm = TRUE)
  sds   <- apply(df_train[, cols, drop = FALSE], 2, sd, na.rm = TRUE)
  sds[sds == 0] <- 1
  return(list(means = means, sds = sds, cols = cols))
}
apply_scaler <- function(df, scaler) {
  df_out <- df
  for (c in scaler$cols) {
    df_out[[c]] <- (df[[c]] - scaler$means[c]) / scaler$sds[c]
  }
  return(df_out)
}
smote_binary_data <- function(df, feat_cols, label_vec, seed = 42) {
  set.seed(seed)
  pos_idx <- which(label_vec == 1)
  neg_idx <- which(label_vec == 0)
  n_pos <- length(pos_idx)
  n_neg <- length(neg_idx)
  if (n_pos == 0 || n_neg == 0 || n_pos == n_neg) return(list(X = as.matrix(df[, feat_cols]), y = label_vec))
  
  if (n_pos < n_neg) {
    minority_idx <- pos_idx
    target_syn   <- n_neg - n_pos
    min_label    <- 1
  } else {
    minority_idx <- neg_idx
    target_syn   <- n_pos - n_neg
    min_label    <- 0
  }
  
  X_min <- as.matrix(df[minority_idx, feat_cols, drop = FALSE])
  syn_matrix <- matrix(0, nrow = target_syn, ncol = length(feat_cols))
  
  for (i in 1:target_syn) {
    base_i <- sample(1:nrow(X_min), 1)
    nn_i   <- sample(1:nrow(X_min), 1)
    alpha  <- runif(1, 0, 1)
    syn_matrix[i, ] <- X_min[base_i, ] + alpha * (X_min[nn_i, ] - X_min[base_i, ])
  }
  
  X_full <- rbind(as.matrix(df[, feat_cols]), syn_matrix)
  y_full <- c(label_vec, rep(min_label, target_syn))
  return(list(X = X_full, y = y_full))
}
compute_cm_metrics <- function(preds, refs) {
  tbl <- table(Prediction = preds, Reference = refs)
  lvls <- levels(refs)
  sens_v <- numeric(length(lvls))
  spec_v <- numeric(length(lvls))
  for (i in seq_along(lvls)) {
    l <- lvls[i]
    tp <- ifelse(l %in% rownames(tbl) && l %in% colnames(tbl), tbl[l, l], 0)
    fn <- sum(tbl[, l]) - tp
    fp <- sum(tbl[l, ]) - tp
    tn <- sum(tbl) - (tp + fn + fp)
    sens_v[i] <- ifelse((tp + fn) > 0, tp / (tp + fn), 0)
    spec_v[i] <- ifelse((tn + fp) > 0, tn / (tn + fp), 0)
  }
  bal_v <- (sens_v + spec_v) / 2
  return(list(table = tbl, Sensitivity = sens_v, Specificity = spec_v, BalancedAccuracy = bal_v))
}
cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data & Construct Expanded 38-Feature Input Matrix
# ---------------------------------------------------------
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) data_file <- paste0("../", data_file)
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
raw_df <- get(data_obj_name, envir = data_env)
target_col_name <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
pulse_last <- get_vec("pulse_last"); pulse_max <- get_vec("pulse_max"); pulse_min <- get_vec("pulse_min")
sbp_last   <- get_vec("sbp_last");   sbp_max   <- get_vec("sbp_max");   sbp_min   <- get_vec("sbp_min")
spo2_last  <- get_vec("spo2_last");  spo2_max  <- get_vec("spo2_max");  spo2_min  <- get_vec("spo2_min")
resp_last  <- get_vec("resp_last");  resp_max  <- get_vec("resp_max");  resp_min  <- get_vec("resp_min")
t_hr       <- get_vec("triage_vital_hr"); t_sbp <- get_vec("triage_vital_sbp"); t_o2 <- get_vec("triage_vital_o2"); t_rr <- get_vec("triage_vital_rr")
hr_rng   <- pulse_max - pulse_min
sbp_rng  <- sbp_max - sbp_min
rr_rng   <- resp_max - resp_min
spo2_rng <- spo2_max - spo2_min
df_master <- data.frame(
  age                     = raw_df$age,
  cc_breathingdifficulty  = cc_bd_vec,
  gender                  = gender_vec,
  triage_vital_hr         = t_hr,
  triage_vital_sbp        = t_sbp,
  triage_vital_rr         = t_rr,
  triage_vital_o2         = t_o2,
  pulse_min               = pulse_min,
  resp_min                = resp_min,
  spo2_min                = spo2_min,
  sbp_min                 = sbp_min,
  pulse_max               = pulse_max,
  resp_max                = resp_max,
  spo2_max                = spo2_max,
  sbp_max                 = sbp_max,
  is_dyspnea_total        = ifelse(t_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(t_o2 > 90 & t_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(t_rr < 10, 1, 0),
  is_tachypnea            = ifelse(t_rr > 30, 1, 0),
  is_hypotension          = ifelse(t_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(t_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(t_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(t_hr > 40 & t_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(t_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(t_hr > 100 & t_hr < 150, 1, 0),
  hr_range                = hr_rng,
  rr_range                = rr_rng,
  spo2_range              = spo2_rng,
  sbp_range               = sbp_rng,
  shock_index             = t_hr / ifelse(t_sbp == 0, 1, t_sbp),
  hr_mid_to_triage        = t_hr - hr_rng,
  sbp_mid_to_triage       = t_sbp - sbp_rng,
  rr_mid_to_triage        = t_rr - rr_rng,
  spo2_mid_to_triage      = t_o2 - spo2_rng,
  rox_index               = t_o2 / ifelse(t_rr == 0, 1, t_rr),
  spo2_drop_ratio         = spo2_rng / ifelse(spo2_max == 0, 1, spo2_max),
  hr_instability_ratio    = hr_rng / (t_hr + 1),
  bif                     = (t_rr / ifelse(t_o2 == 0, 1, t_o2)) * 100
)
layer_feat_names <- names(df_master)
raw_esi <- as.character(raw_df[[target_col_name]])
df_master$target_col <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
df_master <- na.omit(df_master)
test_size <- config$training$test_size
val_size  <- config$training$val_size
in_train_val <- stratified_partition(df_master$target_col, p = 1 - test_size, seed = config$training$random_state)
train_val_df <- df_master[in_train_val, ]
test_df      <- df_master[-in_train_val, ]
rel_val_size <- val_size / (1 - test_size)
in_train    <- stratified_partition(train_val_df$target_col, p = 1 - rel_val_size, seed = config$training$random_state)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]
cont_cols <- c("age", "triage_vital_hr", "triage_vital_sbp", "triage_vital_rr", "triage_vital_o2", "pulse_min", "resp_min", "spo2_min", "sbp_min", "pulse_max", "resp_max", "spo2_max", "sbp_max", "hr_range", "rr_range", "spo2_range", "sbp_range", "shock_index", "hr_mid_to_triage", "sbp_mid_to_triage", "rr_mid_to_triage", "spo2_mid_to_triage", "rox_index", "spo2_drop_ratio", "hr_instability_ratio", "bif")
scaler <- fit_scaler(train_df, cont_cols)
train_scaled <- apply_scaler(train_df, scaler)
val_scaled   <- apply_scaler(val_df, scaler)
test_scaled  <- apply_scaler(test_df, scaler)
cat(sprintf("Partitions Prepared with 38 Uniform Features: Train=%d, Val=%d, Test=%d\n", nrow(train_scaled), nrow(val_scaled), nrow(test_scaled)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Train 3-Tier 4-Model Hierarchical LightGBM Sub-Models (SMOTE)
# ---------------------------------------------------------
set.seed(config$training$random_state)
lgb_params <- list(
  objective        = "binary",
  metric           = "binary_logloss",
  learning_rate    = 0.05,
  num_leaves       = 31,
  max_depth        = 6,
  feature_fraction = 0.8,
  bagging_fraction = 0.8,
  bagging_freq     = 1,
  verbosity        = -1L
)
# Layer 1 (ESI 1 Detector)
smote_l1 <- smote_binary_data(train_scaled, layer_feat_names, ifelse(train_scaled$target_col == "1", 1, 0), seed = config$training$random_state)
dtr_l1   <- lgb.Dataset(smote_l1$X, label = smote_l1$y)
dvl_l1   <- lgb.Dataset(as.matrix(val_scaled[, layer_feat_names]), label = ifelse(val_scaled$target_col == "1", 1, 0))
lgb_l1_m <- lgb.train(params = lgb_params, data = dtr_l1, nrounds = 100, valids = list(val = dvl_l1), early_stopping_rounds = 10, verbose = -1L)
# Layer 2 (ESI 2/3 vs ESI 4/5 Specialist)
tr_l2 <- train_scaled %>% filter(target_col != "1")
vl_l2 <- val_scaled   %>% filter(target_col != "1")
smote_l2 <- smote_binary_data(tr_l2, layer_feat_names, ifelse(tr_l2$target_col %in% c("2", "3"), 1, 0), seed = config$training$random_state)
dtr_l2   <- lgb.Dataset(smote_l2$X, label = smote_l2$y)
dvl_l2   <- lgb.Dataset(as.matrix(vl_l2[, layer_feat_names]), label = ifelse(vl_l2$target_col %in% c("2", "3"), 1, 0))
lgb_l2_m <- lgb.train(params = lgb_params, data = dtr_l2, nrounds = 100, valids = list(val = dvl_l2), early_stopping_rounds = 10, verbose = -1L)
# Layer 3A (ESI 2 vs ESI 3 Specialist)
tr_l3a <- train_scaled %>% filter(target_col %in% c("2", "3"))
vl_l3a <- val_scaled   %>% filter(target_col %in% c("2", "3"))
smote_l3a <- smote_binary_data(tr_l3a, layer_feat_names, ifelse(tr_l3a$target_col == "2", 1, 0), seed = config$training$random_state)
dtr_l3a   <- lgb.Dataset(smote_l3a$X, label = smote_l3a$y)
dvl_l3a   <- lgb.Dataset(as.matrix(vl_l3a[, layer_feat_names]), label = ifelse(vl_l3a$target_col == "2", 1, 0))
lgb_l3a_m <- lgb.train(params = lgb_params, data = dtr_l3a, nrounds = 100, valids = list(val = dvl_l3a), early_stopping_rounds = 10, verbose = -1L)
# Layer 3B (ESI 4 vs ESI 5 Specialist)
tr_l3b <- train_scaled %>% filter(target_col %in% c("4", "5"))
vl_l3b <- val_scaled   %>% filter(target_col %in% c("4", "5"))
smote_l3b <- smote_binary_data(tr_l3b, layer_feat_names, ifelse(tr_l3b$target_col == "4", 1, 0), seed = config$training$random_state)
dtr_l3b   <- lgb.Dataset(smote_l3b$X, label = smote_l3b$y)
dvl_l3b   <- lgb.Dataset(as.matrix(vl_l3b[, layer_feat_names]), label = ifelse(vl_l3b$target_col == "4", 1, 0))
lgb_l3b_m <- lgb.train(params = lgb_params, data = dtr_l3b, nrounds = 100, valids = list(val = dvl_l3b), early_stopping_rounds = 10, verbose = -1L)
# Save RDS Sub-Model Artifacts to deploy/
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)
saveRDS(list(model = lgb_l1_m,  scaler = scaler, is_l1_lgb  = TRUE), file = file.path(deploy_dir, "lightgbm_layer1_esi1_model.rds"))
saveRDS(list(model = lgb_l2_m,  scaler = scaler, is_lgb_l2  = TRUE), file = file.path(deploy_dir, "rf_esi23_esi45_extreme_model.rds"))
saveRDS(list(model = lgb_l3a_m, scaler = scaler, is_lgb_l3a = TRUE), file = file.path(deploy_dir, "lightgbm_esi23_model.rds"))
saveRDS(list(model = lgb_l3b_m, scaler = scaler, is_lgb_l3b = TRUE), file = file.path(deploy_dir, "lightgbm_esi45_model.rds"))
cat("All 4 Sub-Models Trained & Exported to deploy/*.rds!\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Evaluate Combined 3-Tier Soft Probabilistic Pipeline on Test Set
# ---------------------------------------------------------
X_test_mat <- as.matrix(test_scaled[, layer_feat_names])
act_test   <- factor(as.numeric(as.character(test_df$target_col)), levels = 1:5)
p1_test  <- predict(lgb_l1_m,  X_test_mat)
p2_test  <- predict(lgb_l2_m,  X_test_mat)
p3a_test <- predict(lgb_l3a_m, X_test_mat)
p3b_test <- predict(lgb_l3b_m, X_test_mat)
probs_test <- matrix(0, nrow = nrow(test_df), ncol = 5)
probs_test[, 1] <- p1_test
probs_test[, 2] <- (1 - p1_test) * p2_test * p3a_test
probs_test[, 3] <- (1 - p1_test) * p2_test * (1 - p3a_test)
probs_test[, 4] <- (1 - p1_test) * (1 - p2_test) * p3b_test
probs_test[, 5] <- (1 - p1_test) * (1 - p2_test) * (1 - p3b_test)
preds_test <- factor(apply(probs_test, 1, which.max), levels = 1:5)
cm_test    <- compute_cm_metrics(preds_test, act_test)
macro_bal  <- mean(cm_test$BalancedAccuracy)
cat("============================================================\n")
cat("   COMBINED 3-TIER 5-CLASS SOFT PIPELINE TEST REPORT\n")
cat("============================================================\n")
cat(sprintf("  Macro Balanced Accuracy : %.4f\n", macro_bal))
cat("============================================================\n\n")
print(cm_test$table)